<a href="https://colab.research.google.com/github/mehta-ji/ADF/blob/main/hadoopImport.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
sys.path.append('/content/drive/MyDrive/')

In [4]:
import setup_hadoop
print(dir(setup_hadoop))  # You should see 'setup_hadoop' here
setup_hadoop.setup_hadoop()  # call your setup function!

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'os', 'setup_hadoop', 'subprocess']
Hadoop 3.4.1
Source code repository https://github.com/apache/hadoop.git -r 4d7825309348956336b8f06a08322b78422849b1
Compiled by mthakur on 2024-10-09T14:57Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum 7292fe9dba5e2e44e3a9f763fce3e680
This command was run using /content/hadoop-3.4.1/share/hadoop/common/hadoop-common-3.4.1.jar

HADOOP_HOME=/content/hadoop-3.4.1
JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64/


In [5]:
!pip install -q pyspark

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import *

In [7]:
spark = SparkSession.builder.config('spark.ui.port', '4050').getOrCreate()

In [8]:
!hadoop fs -mkdir /input_data

In [9]:
!hadoop fs -put "/content/drive/MyDrive/departments.csv" /input_data

In [10]:
!hadoop fs -ls /input_data

Found 1 items
-rw-r--r--   1 root root        709 2025-08-30 05:50 /input_data/departments.csv


In [11]:
df1 = spark.read.option("header",True).csv("/input_data")

In [12]:
df1.show()

+-------------+--------------------+----------+-----------+
|DEPARTMENT_ID|     DEPARTMENT_NAME|MANAGER_ID|LOCATION_ID|
+-------------+--------------------+----------+-----------+
|           10|      Administration|       200|       1700|
|           20|           Marketing|       201|       1800|
|           30|          Purchasing|       114|       1700|
|           40|     Human Resources|       203|       2400|
|           50|            Shipping|       121|       1500|
|           60|                  IT|       103|       1400|
|           70|    Public Relations|       204|       2700|
|           80|               Sales|       145|       2500|
|           90|           Executive|       100|       1700|
|          100|             Finance|       108|       1700|
|          110|          Accounting|       205|       1700|
|          120|            Treasury|        - |       1700|
|          130|       Corporate Tax|        - |       1700|
|          140|  Control And Credit|    

In [13]:
df1.printSchema()

root
 |-- DEPARTMENT_ID: string (nullable = true)
 |-- DEPARTMENT_NAME: string (nullable = true)
 |-- MANAGER_ID: string (nullable = true)
 |-- LOCATION_ID: string (nullable = true)



In [14]:
df2 = spark.read.csv("/input_data",header=True,inferSchema=True)

df2.printSchema()

root
 |-- DEPARTMENT_ID: integer (nullable = true)
 |-- DEPARTMENT_NAME: string (nullable = true)
 |-- MANAGER_ID: string (nullable = true)
 |-- LOCATION_ID: integer (nullable = true)



In [15]:
!hadoop fs -put '/content/drive/MyDrive/employees.csv' /input_data

In [16]:
df_emp = spark.read.csv("/input_data/employees.csv",header=True,inferSchema=True)
df_emp.printSchema()

root
 |-- EMPLOYEE_ID: integer (nullable = true)
 |-- FIRST_NAME: string (nullable = true)
 |-- LAST_NAME: string (nullable = true)
 |-- EMAIL: string (nullable = true)
 |-- PHONE_NUMBER: string (nullable = true)
 |-- HIRE_DATE: string (nullable = true)
 |-- JOB_ID: string (nullable = true)
 |-- SALARY: integer (nullable = true)
 |-- COMMISSION_PCT: string (nullable = true)
 |-- MANAGER_ID: string (nullable = true)
 |-- DEPARTMENT_ID: integer (nullable = true)



In [17]:
df_emp.select("*").show()

+-----------+----------+---------+--------+------------+---------+----------+------+--------------+----------+-------------+
|EMPLOYEE_ID|FIRST_NAME|LAST_NAME|   EMAIL|PHONE_NUMBER|HIRE_DATE|    JOB_ID|SALARY|COMMISSION_PCT|MANAGER_ID|DEPARTMENT_ID|
+-----------+----------+---------+--------+------------+---------+----------+------+--------------+----------+-------------+
|        198|    Donald| OConnell|DOCONNEL|650.507.9833|21-JUN-07|  SH_CLERK|  2600|            - |       124|           50|
|        199|   Douglas|    Grant|  DGRANT|650.507.9844|13-JAN-08|  SH_CLERK|  2600|            - |       124|           50|
|        200|  Jennifer|   Whalen| JWHALEN|515.123.4444|17-SEP-03|   AD_ASST|  4400|            - |       101|           10|
|        201|   Michael|Hartstein|MHARTSTE|515.123.5555|17-FEB-04|    MK_MAN| 13000|            - |       100|           20|
|        202|       Pat|      Fay|    PFAY|603.123.6666|17-AUG-05|    MK_REP|  6000|            - |       201|           20|


In [18]:
df_emp.select(df_emp.EMPLOYEE_ID,df_emp.FIRST_NAME).show()

+-----------+----------+
|EMPLOYEE_ID|FIRST_NAME|
+-----------+----------+
|        198|    Donald|
|        199|   Douglas|
|        200|  Jennifer|
|        201|   Michael|
|        202|       Pat|
|        203|     Susan|
|        204|   Hermann|
|        205|   Shelley|
|        206|   William|
|        100|    Steven|
|        101|     Neena|
|        102|       Lex|
|        103| Alexander|
|        104|     Bruce|
|        105|     David|
|        106|     Valli|
|        107|     Diana|
|        108|     Nancy|
|        109|    Daniel|
|        110|      John|
+-----------+----------+
only showing top 20 rows



In [19]:
df_emp.select(df_emp["EMPLOYEE_ID"],df_emp["FIRST_NAME"]).show()

+-----------+----------+
|EMPLOYEE_ID|FIRST_NAME|
+-----------+----------+
|        198|    Donald|
|        199|   Douglas|
|        200|  Jennifer|
|        201|   Michael|
|        202|       Pat|
|        203|     Susan|
|        204|   Hermann|
|        205|   Shelley|
|        206|   William|
|        100|    Steven|
|        101|     Neena|
|        102|       Lex|
|        103| Alexander|
|        104|     Bruce|
|        105|     David|
|        106|     Valli|
|        107|     Diana|
|        108|     Nancy|
|        109|    Daniel|
|        110|      John|
+-----------+----------+
only showing top 20 rows



In [20]:
df_emp.select(col("EMPLOYEE_ID").alias("Emp_ID"),col("FIRST_NAME").alias("F_Name")).show()

+------+---------+
|Emp_ID|   F_Name|
+------+---------+
|   198|   Donald|
|   199|  Douglas|
|   200| Jennifer|
|   201|  Michael|
|   202|      Pat|
|   203|    Susan|
|   204|  Hermann|
|   205|  Shelley|
|   206|  William|
|   100|   Steven|
|   101|    Neena|
|   102|      Lex|
|   103|Alexander|
|   104|    Bruce|
|   105|    David|
|   106|    Valli|
|   107|    Diana|
|   108|    Nancy|
|   109|   Daniel|
|   110|     John|
+------+---------+
only showing top 20 rows



In [21]:
df_emp.select(col("EMPLOYEE_ID"),col("FIRST_NAME"),col("EMPLOYEE_ID").alias("Emp_ID")).show()

+-----------+----------+------+
|EMPLOYEE_ID|FIRST_NAME|Emp_ID|
+-----------+----------+------+
|        198|    Donald|   198|
|        199|   Douglas|   199|
|        200|  Jennifer|   200|
|        201|   Michael|   201|
|        202|       Pat|   202|
|        203|     Susan|   203|
|        204|   Hermann|   204|
|        205|   Shelley|   205|
|        206|   William|   206|
|        100|    Steven|   100|
|        101|     Neena|   101|
|        102|       Lex|   102|
|        103| Alexander|   103|
|        104|     Bruce|   104|
|        105|     David|   105|
|        106|     Valli|   106|
|        107|     Diana|   107|
|        108|     Nancy|   108|
|        109|    Daniel|   109|
|        110|      John|   110|
+-----------+----------+------+
only showing top 20 rows



In [22]:
df_emp.select("EMPLOYEE_ID","FIRST_NAME","SALARY").withColumn("NEW_SALARY",col("SALARY") + 1000).show()

+-----------+----------+------+----------+
|EMPLOYEE_ID|FIRST_NAME|SALARY|NEW_SALARY|
+-----------+----------+------+----------+
|        198|    Donald|  2600|      3600|
|        199|   Douglas|  2600|      3600|
|        200|  Jennifer|  4400|      5400|
|        201|   Michael| 13000|     14000|
|        202|       Pat|  6000|      7000|
|        203|     Susan|  6500|      7500|
|        204|   Hermann| 10000|     11000|
|        205|   Shelley| 12008|     13008|
|        206|   William|  8300|      9300|
|        100|    Steven| 24000|     25000|
|        101|     Neena| 17000|     18000|
|        102|       Lex| 17000|     18000|
|        103| Alexander|  9000|     10000|
|        104|     Bruce|  6000|      7000|
|        105|     David|  4800|      5800|
|        106|     Valli|  4800|      5800|
|        107|     Diana|  4200|      5200|
|        108|     Nancy| 12008|     13008|
|        109|    Daniel|  9000|     10000|
|        110|      John|  8200|      9200|
+----------

In [23]:
df_emp.withColumn("NEW_SALARY",col("SALARY") + 1000).select("EMPLOYEE_ID","FIRST_NAME","NEW_SALARY").show()

+-----------+----------+----------+
|EMPLOYEE_ID|FIRST_NAME|NEW_SALARY|
+-----------+----------+----------+
|        198|    Donald|      3600|
|        199|   Douglas|      3600|
|        200|  Jennifer|      5400|
|        201|   Michael|     14000|
|        202|       Pat|      7000|
|        203|     Susan|      7500|
|        204|   Hermann|     11000|
|        205|   Shelley|     13008|
|        206|   William|      9300|
|        100|    Steven|     25000|
|        101|     Neena|     18000|
|        102|       Lex|     18000|
|        103| Alexander|     10000|
|        104|     Bruce|      7000|
|        105|     David|      5800|
|        106|     Valli|      5800|
|        107|     Diana|      5200|
|        108|     Nancy|     13008|
|        109|    Daniel|     10000|
|        110|      John|      9200|
+-----------+----------+----------+
only showing top 20 rows



In [24]:
df_emp.withColumn("NEW_SALARY",col("SALARY") - 1000).select("EMPLOYEE_ID","FIRST_NAME","NEW_SALARY").show()

+-----------+----------+----------+
|EMPLOYEE_ID|FIRST_NAME|NEW_SALARY|
+-----------+----------+----------+
|        198|    Donald|      1600|
|        199|   Douglas|      1600|
|        200|  Jennifer|      3400|
|        201|   Michael|     12000|
|        202|       Pat|      5000|
|        203|     Susan|      5500|
|        204|   Hermann|      9000|
|        205|   Shelley|     11008|
|        206|   William|      7300|
|        100|    Steven|     23000|
|        101|     Neena|     16000|
|        102|       Lex|     16000|
|        103| Alexander|      8000|
|        104|     Bruce|      5000|
|        105|     David|      3800|
|        106|     Valli|      3800|
|        107|     Diana|      3200|
|        108|     Nancy|     11008|
|        109|    Daniel|      8000|
|        110|      John|      7200|
+-----------+----------+----------+
only showing top 20 rows



In [25]:
df_emp.withColumnRenamed("SALARY","EMP_SALARY").show()

+-----------+----------+---------+--------+------------+---------+----------+----------+--------------+----------+-------------+
|EMPLOYEE_ID|FIRST_NAME|LAST_NAME|   EMAIL|PHONE_NUMBER|HIRE_DATE|    JOB_ID|EMP_SALARY|COMMISSION_PCT|MANAGER_ID|DEPARTMENT_ID|
+-----------+----------+---------+--------+------------+---------+----------+----------+--------------+----------+-------------+
|        198|    Donald| OConnell|DOCONNEL|650.507.9833|21-JUN-07|  SH_CLERK|      2600|            - |       124|           50|
|        199|   Douglas|    Grant|  DGRANT|650.507.9844|13-JAN-08|  SH_CLERK|      2600|            - |       124|           50|
|        200|  Jennifer|   Whalen| JWHALEN|515.123.4444|17-SEP-03|   AD_ASST|      4400|            - |       101|           10|
|        201|   Michael|Hartstein|MHARTSTE|515.123.5555|17-FEB-04|    MK_MAN|     13000|            - |       100|           20|
|        202|       Pat|      Fay|    PFAY|603.123.6666|17-AUG-05|    MK_REP|      6000|         

In [26]:
df_emp.drop("COMMISSION_PCT","SALARY").show()

+-----------+----------+---------+--------+------------+---------+----------+----------+-------------+
|EMPLOYEE_ID|FIRST_NAME|LAST_NAME|   EMAIL|PHONE_NUMBER|HIRE_DATE|    JOB_ID|MANAGER_ID|DEPARTMENT_ID|
+-----------+----------+---------+--------+------------+---------+----------+----------+-------------+
|        198|    Donald| OConnell|DOCONNEL|650.507.9833|21-JUN-07|  SH_CLERK|       124|           50|
|        199|   Douglas|    Grant|  DGRANT|650.507.9844|13-JAN-08|  SH_CLERK|       124|           50|
|        200|  Jennifer|   Whalen| JWHALEN|515.123.4444|17-SEP-03|   AD_ASST|       101|           10|
|        201|   Michael|Hartstein|MHARTSTE|515.123.5555|17-FEB-04|    MK_MAN|       100|           20|
|        202|       Pat|      Fay|    PFAY|603.123.6666|17-AUG-05|    MK_REP|       201|           20|
|        203|     Susan|   Mavris| SMAVRIS|515.123.7777|07-JUN-02|    HR_REP|       101|           40|
|        204|   Hermann|     Baer|   HBAER|515.123.8888|07-JUN-02|    PR_

In [27]:
df_emp.where(col("SALARY") < 2500).show()

+-----------+----------+----------+--------+------------+---------+--------+------+--------------+----------+-------------+
|EMPLOYEE_ID|FIRST_NAME| LAST_NAME|   EMAIL|PHONE_NUMBER|HIRE_DATE|  JOB_ID|SALARY|COMMISSION_PCT|MANAGER_ID|DEPARTMENT_ID|
+-----------+----------+----------+--------+------------+---------+--------+------+--------------+----------+-------------+
|        127|     James|    Landry| JLANDRY|650.124.1334|14-JAN-07|ST_CLERK|  2400|            - |       120|           50|
|        128|    Steven|    Markle| SMARKLE|650.124.1434|08-MAR-08|ST_CLERK|  2200|            - |       120|           50|
|        132|        TJ|     Olson| TJOLSON|650.124.8234|10-APR-07|ST_CLERK|  2100|            - |       121|           50|
|        135|        Ki|       Gee|    KGEE|650.127.1734|12-DEC-07|ST_CLERK|  2400|            - |       122|           50|
|        136|     Hazel|Philtanker|HPHILTAN|650.127.1634|06-FEB-08|ST_CLERK|  2200|            - |       122|           50|
+-------

In [28]:
df_emp.filter((col("SALARY") < 5000) & (col("DEPARTMENT_ID") == 50)).show()

+-----------+----------+-----------+--------+------------+---------+--------+------+--------------+----------+-------------+
|EMPLOYEE_ID|FIRST_NAME|  LAST_NAME|   EMAIL|PHONE_NUMBER|HIRE_DATE|  JOB_ID|SALARY|COMMISSION_PCT|MANAGER_ID|DEPARTMENT_ID|
+-----------+----------+-----------+--------+------------+---------+--------+------+--------------+----------+-------------+
|        198|    Donald|   OConnell|DOCONNEL|650.507.9833|21-JUN-07|SH_CLERK|  2600|            - |       124|           50|
|        199|   Douglas|      Grant|  DGRANT|650.507.9844|13-JAN-08|SH_CLERK|  2600|            - |       124|           50|
|        125|     Julia|      Nayer|  JNAYER|650.124.1214|16-JUL-05|ST_CLERK|  3200|            - |       120|           50|
|        126|     Irene|Mikkilineni|IMIKKILI|650.124.1224|28-SEP-06|ST_CLERK|  2700|            - |       120|           50|
|        127|     James|     Landry| JLANDRY|650.124.1334|14-JAN-07|ST_CLERK|  2400|            - |       120|           50|


In [29]:
df_emp.where("SALARY < 5000 and DEPARTMENT_ID != 50").show()

+-----------+----------+----------+--------+------------+---------+--------+------+--------------+----------+-------------+
|EMPLOYEE_ID|FIRST_NAME| LAST_NAME|   EMAIL|PHONE_NUMBER|HIRE_DATE|  JOB_ID|SALARY|COMMISSION_PCT|MANAGER_ID|DEPARTMENT_ID|
+-----------+----------+----------+--------+------------+---------+--------+------+--------------+----------+-------------+
|        200|  Jennifer|    Whalen| JWHALEN|515.123.4444|17-SEP-03| AD_ASST|  4400|            - |       101|           10|
|        105|     David|    Austin| DAUSTIN|590.423.4569|25-JUN-05| IT_PROG|  4800|            - |       103|           60|
|        106|     Valli| Pataballa|VPATABAL|590.423.4560|05-FEB-06| IT_PROG|  4800|            - |       103|           60|
|        107|     Diana|   Lorentz|DLORENTZ|590.423.5567|07-FEB-07| IT_PROG|  4200|            - |       103|           60|
|        115| Alexander|      Khoo|   AKHOO|515.127.4562|18-MAY-03|PU_CLERK|  3100|            - |       114|           30|
|       

In [30]:
df_emp.distinct().show(100)

+-----------+-----------+-----------+--------+------------+---------+----------+------+--------------+----------+-------------+
|EMPLOYEE_ID| FIRST_NAME|  LAST_NAME|   EMAIL|PHONE_NUMBER|HIRE_DATE|    JOB_ID|SALARY|COMMISSION_PCT|MANAGER_ID|DEPARTMENT_ID|
+-----------+-----------+-----------+--------+------------+---------+----------+------+--------------+----------+-------------+
|        120|    Matthew|      Weiss|  MWEISS|650.123.1234|18-JUL-04|    ST_MAN|  8000|            - |       100|           50|
|        118|        Guy|     Himuro| GHIMURO|515.127.4565|15-NOV-06|  PU_CLERK|  2600|            - |       114|           30|
|        110|       John|       Chen|   JCHEN|515.124.4269|28-SEP-05|FI_ACCOUNT|  8200|            - |       108|          100|
|        123|     Shanta|    Vollman|SVOLLMAN|650.123.4234|10-OCT-05|    ST_MAN|  6500|            - |       100|           50|
|        124|      Kevin|    Mourgos|KMOURGOS|650.123.5234|16-NOV-07|    ST_MAN|  5800|            - |  

In [31]:
df_emp.dropDuplicates(["DEPARTMENT_ID","HIRE_DATE"]).select("EMPLOYEE_ID","HIRE_DATE","DEPARTMENT_ID").orderBy("EMPLOYEE_ID").show(100)

+-----------+---------+-------------+
|EMPLOYEE_ID|HIRE_DATE|DEPARTMENT_ID|
+-----------+---------+-------------+
|        100|17-JUN-03|           90|
|        101|21-SEP-05|           90|
|        102|13-JAN-01|           90|
|        103|03-JAN-06|           60|
|        104|21-MAY-07|           60|
|        105|25-JUN-05|           60|
|        106|05-FEB-06|           60|
|        107|07-FEB-07|           60|
|        108|17-AUG-02|          100|
|        109|16-AUG-02|          100|
|        110|28-SEP-05|          100|
|        111|30-SEP-05|          100|
|        112|07-MAR-06|          100|
|        113|07-DEC-07|          100|
|        114|07-DEC-02|           30|
|        115|18-MAY-03|           30|
|        116|24-DEC-05|           30|
|        117|24-JUL-05|           30|
|        118|15-NOV-06|           30|
|        119|10-AUG-07|           30|
|        120|18-JUL-04|           50|
|        121|10-APR-05|           50|
|        122|01-MAY-03|           50|
|        123

In [32]:
df_emp.select(count("SALARY")).show()

+-------------+
|count(SALARY)|
+-------------+
|           50|
+-------------+



In [33]:
df_emp.select(count("SALARY").alias("total_count")).show()

+-----------+
|total_count|
+-----------+
|         50|
+-----------+



In [34]:
df_emp.select(count("*").alias("total_count")).show()

+-----------+
|total_count|
+-----------+
|         50|
+-----------+



In [35]:
df_emp.select(max("SALARY").alias("max_salary")).show()

+----------+
|max_salary|
+----------+
|     24000|
+----------+



In [36]:
df_emp.select(min("SALARY").alias("min_salary")).show()

+----------+
|min_salary|
+----------+
|      2100|
+----------+



In [37]:
df_emp.select(avg("SALARY").alias("avg_salary")).show()

+----------+
|avg_salary|
+----------+
|   6182.32|
+----------+



In [38]:
df_emp.groupBy("DEPARTMENT_ID").agg(sum("SALARY").alias("avg_salary"),count("*")).show()

+-------------+----------+--------+
|DEPARTMENT_ID|avg_salary|count(1)|
+-------------+----------+--------+
|           20|     19000|       2|
|           40|      6500|       1|
|          100|     51608|       6|
|           10|      4400|       1|
|           50|     85600|      23|
|           70|     10000|       1|
|           90|     58000|       3|
|           60|     28800|       5|
|          110|     20308|       2|
|           30|     24900|       6|
+-------------+----------+--------+



In [39]:
df_emp.where(col("DEPARTMENT_ID")==50).select(count("DEPARTMENT_ID")).show()

+--------------------+
|count(DEPARTMENT_ID)|
+--------------------+
|                  23|
+--------------------+



In [40]:
df_emp.orderBy(col("SALARY").desc(),col("DEPARTMENT_ID").asc()).select("EMPLOYEE_ID","FIRST_NAME","DEPARTMENT_ID","SALARY").show()

+-----------+-----------+-------------+------+
|EMPLOYEE_ID| FIRST_NAME|DEPARTMENT_ID|SALARY|
+-----------+-----------+-------------+------+
|        100|     Steven|           90| 24000|
|        101|      Neena|           90| 17000|
|        102|        Lex|           90| 17000|
|        201|    Michael|           20| 13000|
|        108|      Nancy|          100| 12008|
|        205|    Shelley|          110| 12008|
|        114|        Den|           30| 11000|
|        204|    Hermann|           70| 10000|
|        103|  Alexander|           60|  9000|
|        109|     Daniel|          100|  9000|
|        206|    William|          110|  8300|
|        121|       Adam|           50|  8200|
|        110|       John|          100|  8200|
|        120|    Matthew|           50|  8000|
|        122|      Payam|           50|  7900|
|        112|Jose Manuel|          100|  7800|
|        111|     Ismael|          100|  7700|
|        113|       Luis|          100|  6900|
|        203|

In [41]:
df_emp.groupBy(col("DEPARTMENT_ID")).agg(sum("SALARY").alias("SUM_SALARY"), max("SALARY").alias("MAX_SALARY"), min("SALARY").alias("MIN_SALARY"), avg("SALARY").alias("AVG_SALARY")).show()

+-------------+----------+----------+----------+------------------+
|DEPARTMENT_ID|SUM_SALARY|MAX_SALARY|MIN_SALARY|        AVG_SALARY|
+-------------+----------+----------+----------+------------------+
|           20|     19000|     13000|      6000|            9500.0|
|           40|      6500|      6500|      6500|            6500.0|
|          100|     51608|     12008|      6900| 8601.333333333334|
|           10|      4400|      4400|      4400|            4400.0|
|           50|     85600|      8200|      2100|3721.7391304347825|
|           70|     10000|     10000|     10000|           10000.0|
|           90|     58000|     24000|     17000|19333.333333333332|
|           60|     28800|      9000|      4200|            5760.0|
|          110|     20308|     12008|      8300|           10154.0|
|           30|     24900|     11000|      2500|            4150.0|
+-------------+----------+----------+----------+------------------+



In [43]:
#Equivalent to having, in PySpark, a combination of groupBy and where forms Having
df_emp.groupBy(col("DEPARTMENT_ID")).agg(sum("SALARY").alias("SUM_SALARY"), max("SALARY").alias("MAX_SALARY"), min("SALARY").alias("MIN_SALARY"), avg("SALARY").alias("AVG_SALARY")).where(col("MAX_SALARY") >=10000).show()

+-------------+----------+----------+----------+------------------+
|DEPARTMENT_ID|SUM_SALARY|MAX_SALARY|MIN_SALARY|        AVG_SALARY|
+-------------+----------+----------+----------+------------------+
|           20|     19000|     13000|      6000|            9500.0|
|          100|     51608|     12008|      6900| 8601.333333333334|
|           70|     10000|     10000|     10000|           10000.0|
|           90|     58000|     24000|     17000|19333.333333333332|
|          110|     20308|     12008|      8300|           10154.0|
|           30|     24900|     11000|      2500|            4150.0|
+-------------+----------+----------+----------+------------------+



In [ ]:
df_emp.